In [2]:
!pip install mediapipe opencv-python scikit-learn matplotlib numpy tensorflow


In [3]:
import cv2
import numpy as np
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,  
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)


In [4]:
import os
GESTURES = ["HELLO", "YES", "NO"]
DATA_PATH = "gesture_data"
os.makedirs(DATA_PATH, exist_ok=True)


In [5]:
def collect_data(gesture_name, num_samples=100):
    cap = cv2.VideoCapture(0)
    collected = 0
    print(f"Collecting: {gesture_name}")

    while collected < num_samples:
        success, frame = cap.read()
        if not success:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(frame_rgb)

        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                landmarks = []
                for lm in hand_landmarks.landmark:
                    landmarks.extend([lm.x, lm.y, lm.z])  

                npy_path = os.path.join(DATA_PATH, f"{gesture_name}_{collected}.npy")
                np.save(npy_path, landmarks)
                
                collected += 1
                cv2.putText(frame, f"{gesture_name} Data: {collected}/{num_samples}",
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

        cv2.imshow("Collecting Data", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Done!")


In [6]:
collect_data("HELLO")


Collecting: HELLO
Done!


In [7]:
collect_data("YES")


Collecting: YES
Done!


In [8]:
collect_data("NO")


Collecting: NO
Done!


In [9]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu


In [10]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


GESTURES = ["HELLO", "YES", "NO"]
DATA_PATH = "gesture_data"

X = []
y = []

for fname in sorted(os.listdir(DATA_PATH)):
    for label, gesture in enumerate(GESTURES):
        if fname.startswith(gesture):
            arr = np.load(os.path.join(DATA_PATH, fname))
            arr = np.array(arr).flatten()     # ✅ flatten always
            X.append(arr)
            y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("✅ Data Loaded:", X.shape, "Labels:", y.shape)

# torch tensors
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

input_size = X.shape[1]
num_classes = len(GESTURES)

print("Input size:", input_size)

# ✅ Simple clean model
class SignModel(nn.Module):
    def __init__(self):
        super(SignModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SignModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ✅ Training loop
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        pred = model(batch_x)
        loss = criterion(pred, batch_y.long())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

# ✅ Testing score
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        output = model(batch_x)
        preds = torch.argmax(output, dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

print(f"✅ Test Accuracy: {correct/total:.2f}")

# ✅ Save model
torch.save(model.state_dict(), "sign_model.pth")
print("✔️ Model saved: sign_model.pth")


✅ Data Loaded: (300, 63) Labels: (300,)
Input size: 63
Epoch 1/20, Loss: 16.1403
Epoch 2/20, Loss: 14.7128
Epoch 3/20, Loss: 13.3121
Epoch 4/20, Loss: 11.6879
Epoch 5/20, Loss: 9.8600
Epoch 6/20, Loss: 8.1150
Epoch 7/20, Loss: 6.6705
Epoch 8/20, Loss: 5.4793
Epoch 9/20, Loss: 4.5762
Epoch 10/20, Loss: 3.9055
Epoch 11/20, Loss: 3.4586
Epoch 12/20, Loss: 3.0048
Epoch 13/20, Loss: 2.7338
Epoch 14/20, Loss: 2.5116
Epoch 15/20, Loss: 2.3265
Epoch 16/20, Loss: 2.2127
Epoch 17/20, Loss: 2.1231
Epoch 18/20, Loss: 2.0332
Epoch 19/20, Loss: 1.9654
Epoch 20/20, Loss: 1.9337
✅ Test Accuracy: 0.98
✔️ Model saved: sign_model.pth


In [11]:
!pip install gTTS playsound


In [12]:
pip install pyttsx3==2.90


Note: you may need to restart the kernel to use updated packages.


In [13]:
import pyttsx3
engine = pyttsx3.init()
engine.say("Testing voice output. If you hear this, it works.")
engine.runAndWait()
print("✅ Voice test completed")


✅ Voice test completed


In [14]:
!pip install SpeechRecognition
!pip install pyaudio


In [ ]:
import speech_recognition as sr
import time

# Extended gesture dictionary with synonyms
GESTURE_SIGNS = {
    "hello": "👋",
    "hi": "👋",
    "hey": "👋",
    "yes": "👍",
    "yeah": "👍",
    "sure": "👍",
    "no": "👎",
    "nope": "👎",
    "nah": "👎"
}

def speech_to_sign():
    recognizer = sr.Recognizer()
    mic = sr.Microphone()
    
    print("🎙️ Speak now... (say Hello / Yes / No)")
    
    with mic as source:
        recognizer.adjust_for_ambient_noise(source)
        print("🕒 Listening...")
        audio = recognizer.listen(source, timeout=3, phrase_time_limit=4)
        
    try:
        text = recognizer.recognize_google(audio).lower().strip()
        print(f"🗣️ You said: {text}")

        # find the first matching keyword
        detected = None
        for word, sign in GESTURE_SIGNS.items():
            if word in text:
                detected = (word, sign)
                break

        if detected:
            print(f"🤟 Detected Sign: {detected[1]} ({detected[0].upper()})")
        else:
            print("⚠️ No matching sign found.")

    except sr.UnknownValueError:
        print("❌ Could not understand audio.")
    except sr.RequestError:
        print("⚠️ Speech service unavailable.")

# Run continuously (optional)
while True:
    speech_to_sign()
    print("\nSay something again in 3 seconds...\n")
    time.sleep(3)


In [ ]:
import win32com.client as wincl

speaker = wincl.Dispatch("SAPI.SpVoice")
speaker.Voice = speaker.GetVoices().Item(1)  # Select Zira (female)
speaker.Speak("Hello, this is Zira. Your gesture control voice is now active.")

In [18]:
import cv2
import mediapipe as mp
import win32com.client as wincl
import torch
import numpy as np
import time

# ✅ Speech setup
speaker = wincl.Dispatch("SAPI.SpVoice")

# ✅ Gestures & Model
GESTURES = ["HELLO", "YES", "NO"]

class SignModel(torch.nn.Module):
    def __init__(self):
        super(SignModel, self).__init__()
        self.fc1 = torch.nn.Linear(63, 128)
        self.fc2 = torch.nn.Linear(128, len(GESTURES))

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SignModel()
model.load_state_dict(torch.load("sign_model.pth", map_location="cpu"))
model.eval()

# ✅ Mediapipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1)
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
last_gesture = None
last_speak_time = 0

def speak(text):
    print("🎙 Speaking:", text)
    speaker.Speak(text)
    time.sleep(0.3)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    image = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(image, hand, mp_hands.HAND_CONNECTIONS)

            # ✅ Landmark extraction
            landmarks = []
            for lm in hand.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            inp = torch.tensor([landmarks], dtype=torch.float32)
            output = model(inp)
            pred = torch.argmax(output, dim=1).item()

            gesture = GESTURES[pred]

            # ✅ Display on screen
            cv2.putText(image, gesture, (30, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.3, (0, 255, 0), 3)

            # ✅ Speak only when gesture changes
            if gesture != last_gesture and time.time() - last_speak_time > 1:
                speak(gesture)
                last_gesture = gesture
                last_speak_time = time.time()

    cv2.imshow("Sign Detection", image)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


🎙 Speaking: NO
🎙 Speaking: YES
🎙 Speaking: NO
🎙 Speaking: HELLO
🎙 Speaking: NO
🎙 Speaking: YES
🎙 Speaking: NO


KeyboardInterrupt: 